In [2]:
import cv2, torch
import numpy as np

model_path = "../models/face_detection_yunet_2023mar.onnx"
image_path = "../assets/images/street1.jpg"

image = cv2.imread(image_path)

detector = cv2.FaceDetectorYN.create(
    model_path,
    "",
    (320, 320),
    0.6,
    0.3,
    5000
)

h, w = image.shape[:2]
detector.setInputSize((w, h))

_, faces = detector.detect(image)

# Standard ArcFace 112x112 landmark template
arcface_template = np.array([
    [38.2946, 51.6963],  # left eye
    [73.5318, 51.5014],  # right eye
    [56.0252, 71.7366],  # nose
    [41.5493, 92.3655],  # left mouth
    [70.7299, 92.2041]   # right mouth
], dtype=np.float32)

def get_emb(face):
    aligned_face = np.transpose(face, (2, 0, 1))
    aligned_face = np.expand_dims(aligned_face, axis=0)
    aligned_face = torch.from_numpy(aligned_face).float()

    aligned_face = (aligned_face - 127.5) / 127.5
    
    embs = model(aligned_face)
    embs = F.normalize(embs, dim=1)
    return embs

if faces is not None:
    for i, face in enumerate(faces):
        landmarks = face[4:14].reshape(5, 2).astype(np.float32)
        transform, _ = cv2.estimateAffinePartial2D(
            landmarks,
            arcface_template,
            method=cv2.LMEDS
        )
    
        # Align face to 112x112
        aligned_face = cv2.warpAffine(
            image,
            transform,
            (112, 112),
            flags=cv2.INTER_LINEAR
        )

        # Estimate similarity transform
        transform, _ = cv2.estimateAffinePartial2D(
            landmarks,
            arcface_template,
            method=cv2.LMEDS
        )

        # Align face to 112x112
        aligned_face = cv2.warpAffine(
            image,
            transform,
            (112, 112),
            flags=cv2.INTER_LINEAR
        )
        break

print("Done")

Done


In [3]:
import timm
import torch.nn.functional as F

model = timm.create_model("hf_hub:gaunernst/vit_tiny_patch8_112.arcface_ms1mv3", pretrained=True).eval()

In [16]:
dhoni_1 = cv2.imread("assets/dhoni_1.jpg")
dhoni_2 = cv2.imread("assets/dhoni_2.jpg")
# me = cv2.resize(cv2.imread("../assets/images/2.jpg"), dsize=(112, 112), interpolation=cv2.INTER_CUBIC)
# ajay = cv2.resize(cv2.imread("../assets/images/1.jpg"), dsize=(112, 112), interpolation=cv2.INTER_CUBIC)

me = cv2.imread("../assets/images/2.jpg")
ajay = cv2.imread("../assets/images/1.jpg")

embb = torch.Tensor(np.loadtxt("../assets/emb/modi", dtype=np.float32))

arr = []
for i in [ajay, me]:
    h, w = i.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(i)
    if len(faces) > 0:
        landmarks = faces[0][4:14].reshape(5, 2).astype(np.float32)
        transform, _ = cv2.estimateAffinePartial2D(
            landmarks,
            arcface_template,
            method=cv2.LMEDS
        )
    
        # Align face to 112x112
        aligned_face = cv2.warpAffine(
            i,
            transform,
            (112, 112),
            flags=cv2.INTER_LINEAR
        )

        # Estimate similarity transform
        transform, _ = cv2.estimateAffinePartial2D(
            landmarks,
            arcface_template,
            method=cv2.LMEDS
        )

        # Align face to 112x112
        aligned_face = cv2.warpAffine(
            i,
            transform,
            (112, 112),
            flags=cv2.INTER_LINEAR
        )
        arr.append(get_emb(aligned_face))

similarity = torch.sum(embb * arr[1], dim=1)
similarity.item()

0.24858474731445312

In [7]:
import os, cv2
for i in os.listdir("faces"):
    image = cv2.imread(f"faces/{i}")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    fm = cv2.Laplacian(gray, cv2.CV_64F).var()
    print(i, fm)

0.jpg 649.2453783856571
1.jpg 692.0514741628299
5.jpg 1246.544292355372
4.jpg 358.0513800669218
3.jpg 3443.385802469136
2.jpg 528.9882917820868


In [2]:
from ultralytics import YOLOWorld

# 1. Load the model
model = YOLOWorld("../models/yolov8s-worldv2.pt")

# 2. Add prompt variations to improve CLIP embedding match
model.set_classes(["spoon", "metal spoon", "silver spoon", "eating utensil", "cutlery"])

# 3. Run with higher input resolution and an ultra-low threshold
results = model.predict(
    "../assets/images/kitchen.jpg",
    conf=0.03,         # Drop to 0.03 to inspect raw candidates
    imgsz=1280,        # CRITICAL: Default 640 shrinks a spoon into ~5-10 pixels
    save=True
)

# 4. Inspect raw detection tensors directly
boxes = results[0].boxes
print(f"Total detections found: {len(boxes)}")

for box in boxes:
    cls_id = int(box.cls[0])
    score = float(box.conf[0])
    label = results[0].names[cls_id]
    coords = [round(x, 1) for x in box.xyxy[0].tolist()]
    print(f"Found '{label}' | Confidence: {score:.3f} | BBox: {coords}")
results[0].show()


image 1/1 C:\Users\ajayk\OneDrive\Desktop\Amar\AI\Projects\Vissora\Server\..\assets\images\kitchen.jpg: 1280x1280 7 metal spoons, 8 eating utensils, 18 cutlerys, 769.1ms
Speed: 13.2ms preprocess, 769.1ms inference, 1.8ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to C:\Users\ajayk\OneDrive\Desktop\Amar\AI\Projects\Vissora\Server\runs\detect\predict-2
Total detections found: 33
Found 'cutlery' | Confidence: 0.748 | BBox: [337.7, 538.5, 518.2, 681.2]
Found 'metal spoon' | Confidence: 0.720 | BBox: [231.9, 554.7, 387.6, 654.4]
Found 'metal spoon' | Confidence: 0.708 | BBox: [128.8, 516.2, 310.9, 609.3]
Found 'cutlery' | Confidence: 0.617 | BBox: [423.7, 572.4, 611.3, 705.5]
Found 'cutlery' | Confidence: 0.473 | BBox: [382.4, 546.8, 610.1, 701.8]
Found 'cutlery' | Confidence: 0.471 | BBox: [235.4, 554.1, 386.6, 653.6]
Found 'cutlery' | Confidence: 0.349 | BBox: [339.3, 549.9, 588.9, 685.7]
Found 'cutlery' | Confidence: 0.347 | BBox: [133.3, 515.5, 310.0, 601.7]
Found 